# DeepSeek V4.1 Flash: three-mode validation comparison

This notebook compares **non-thinking**, **thinking-low** and **thinking-high** API configurations on identical NYT10m supplied-entity bags. The default pilot uses the existing confirmed 30 and fixed random 100 validation bags (130 distinct pairs). Current human annotations and distant-supervision labels are scored separately. Saved local encoder scores provide additional comparators without loading or running models.


**Protocol v2 (7 October):** all modes receive the same explicit positive JSON syntax example. The original v1 results remain unchanged. The new prompt uses a separate fingerprint and cache directory, so the default pilot makes up to **390 new requests (130 per mode)**, excluding transport retries. Resuming v2 reuses its completed responses.
All experiment functions are defined in this notebook. It uses Python's standard-library HTTPS client and the project's existing pandas installation. It installs no packages and downloads no model weights. API inference requires an internet connection.

**Execution:** add `DEEPSEEK_API_KEY=...` to the ignored project `.env`, or set that environment variable. Select the project's `.venv` kernel. Set `RUN_API=True` in section 1, then use **Run All**. With `RUN_API=False`, only local preparation and analysis occur; an API key is not requested. API results are resumable and stored under the ignored `data/pilot_runs/` directory.

**Scope:** all three pilot modes are enabled by default, with a $2 model-usage budget. The optional `full_validation` scope reuses completed pilot calls with identical settings. It requires an explicitly increased budget; no official test inputs are read. The model, prompt, decoding, parser and failure rules remain development configurations until frozen.

**Sources:** [pricing](https://api-docs.deepseek.com/quick_start/pricing/), [thinking controls](https://api-docs.deepseek.com/guides/thinking_mode/), [Chat Completions API](https://api-docs.deepseek.com/api/create-chat-completion/), [JSON output](https://api-docs.deepseek.com/guides/json_mode/), [released weights](https://huggingface.co/deepseek-ai/DeepSeek-V4.1-Flash). The API identifier `deepseek-flash` currently advertises V4.1 Flash; it is a service alias, not a cryptographically pinned checkpoint. Returned model IDs and backend fingerprints are retained.

This development notebook requires the retained local validation inputs and caches to execute. Its saved results and compact exports are included for review; paid calls default to disabled.


## 1. Imports and execution settings

Flags control paid calls, scope, configurations, off-peak waiting and the model-usage budget. All three configurations use the same prompt, evidence and 4,096-token total completion limit. That limit includes reasoning when enabled; it is not a 1,000-token reasoning allocation. Truncated outputs are recorded as invalid. Sampling parameters are omitted in all three conditions: the provider does not support temperature control in thinking mode, so this is not a deterministic temperature-matched experiment.

Requests are sequential. The first configuration rotates across bags to reduce a fixed order effect. Reported API time includes network/service latency and is distinct from local CPU inference time. No tools, ICL demonstrations or reference answers are sent.

In [8]:
import ast
import contextlib
import hashlib
import itertools
import json
import math
import os
import random
import shlex
import shutil
import socket
import subprocess
import sys
import tempfile
import time
import urllib.error
import urllib.request
import uuid
from collections import Counter, defaultdict
from datetime import datetime, timedelta, timezone
from getpass import getpass
from pathlib import Path
from zoneinfo import ZoneInfo

import pandas as pd
from IPython.display import display

# Locate the checkout from either its root or source directory.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'results/full_validation_settings.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook from the Ensembling checkout.')

RUN_API = False
SCOPE = 'pilot'                       # 'pilot' or 'full_validation'
RUN_NON_THINKING = True
RUN_THINKING_LOW = True
RUN_THINKING_HIGH = True
MAX_COMPLETION_TOKENS = 4096
MAX_BUDGET_USD = 2.00                 # includes saved calls and conservative unknown-charge reserves
OFF_PEAK_ONLY = False                # submit immediately in either tariff window
WAIT_FOR_OFF_PEAK = False            # used only when OFF_PEAK_ONLY=True
BOUNDARY_BUFFER_SECONDS = 300
REQUEST_TIMEOUT_SECONDS = 180
ALLOW_RESUBMIT_UNCERTAIN_ATTEMPT_IDS = ()  # only after account-side review; unknown-charge reserves remain
MAX_HTTP_RETRIES = 2                 # only explicit 429/502/503/504 responses
STOP_AFTER_CONSECUTIVE_FAILURES = 3
KEEP_AWAKE = True                    # macOS caffeinate during API execution
COMPARE_ENCODER_CACHES = True
ENCODER_THRESHOLDS = (0.5, 0.7, 0.9)
MAX_REVIEWED_ERRORS_TO_DISPLAY = 30

MODEL_ID = 'deepseek-flash'
ADVERTISED_RELEASE = 'DeepSeek-V4.1-Flash'
API_URL = 'https://api.deepseek.com/chat/completions'
PROTOCOL_VERSION = 'deepseek_three_mode_validation_v2_20261007'
RUN_DIRECTORY_NAME = 'deepseek_v41_flash_validation_v2_20261007'
PREVIOUS_PILOT_REPORT = ROOT / 'results/deepseek_flash_pilot_summary_d17dec454f4f71a9.json'
PRICE_VERSION = 'deepseek_flash_pricing_20261006'
RATES = {
    'off_peak': {'input_miss': 0.15, 'input_hit': 0.003, 'output': 0.60},
    'peak': {'input_miss': 0.30, 'input_hit': 0.006, 'output': 1.20},
}
RELEX_RUN = ROOT / 'data/pilot_runs/relex_full_validation_given_entities_v1_20261004'
GLIDRE_RUN = ROOT / 'data/pilot_runs/glidre_full_validation_given_entities_v1_20261006'
RANDOM_100 = ROOT / 'data/reduction_pilot/validation_100_20261006'
NOTEBOOK_PATH = ROOT / 'source/DeepSeek_flash_validation.ipynb'
ROME = ZoneInfo('Europe/Rome')

if SCOPE not in {'pilot', 'full_validation'}:
    raise ValueError('Invalid scope.')
if not (RUN_NON_THINKING or RUN_THINKING_LOW or RUN_THINKING_HIGH) or MAX_BUDGET_USD <= 0:
    raise ValueError('Enable at least one configuration and a positive budget.')
print('Kernel:', sys.executable)
print('Scope:', SCOPE, '| paid calls enabled:', RUN_API)

Kernel: /Users/alessandro/Progetti/Ensembling/.venv/bin/python
Scope: pilot | paid calls enabled: True


## 2. Frozen inputs and current references

A bag is an ordered head → tail pair plus its selected news sentence records. Entity character spans are zero-based, half-open offsets in the original sentence. The existing selection retains at most 20 records, including duplicate texts. The API receives the same evidence and supplied mentions as the encoder experiment.

This cell checks the original inference fingerprint and source hashes, all supplied spans, the fixed random100 provenance and the current confirmed C/N annotation files. Reviewed references are loaded separately from requests. Annotation edits change report provenance, not completed API predictions. The 30 are a selected development diagnostic; random100 distant labels are noisy and are not manually verified gold.

In [9]:
def canonical(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':'), allow_nan=False)

def digest(value):
    return hashlib.sha256(canonical(value).encode('utf-8')).hexdigest()

def file_digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def input_only(bag):
    fields = ('pair_ids', 'head', 'tail', 'sentences', 'entity_positions',
              'full_bag_size', 'selected_indices_zero_based')
    return {key: bag[key] for key in fields}

frozen = json.loads((RELEX_RUN / 'settings.json').read_text())
published = json.loads((ROOT / 'results/full_validation_settings.json').read_text())
expected_fp = hashlib.sha256(json.dumps(frozen['settings'], sort_keys=True).encode()).hexdigest()
if frozen['fingerprint'] != expected_fp or frozen != published:
    raise RuntimeError('Local/published frozen settings do not agree.')
for key, relative in {
    'runner_sha256': 'source/run_full_validation_relex_given_entities.py',
    'adapter_sha256': 'source/relex_given_entities.py',
    'label_module_sha256': 'source/run_extended_gliner_relex.py',
}.items():
    if file_digest(ROOT / relative) != frozen['settings'][key]:
        raise RuntimeError(f'Frozen inference source changed: {relative}')

LABEL_DESCRIPTIONS = frozen['settings']['label_name']
ALLOWED = set(LABEL_DESCRIPTIONS)
if len(ALLOWED) != 24 or 'NA' in ALLOWED:
    raise RuntimeError('Expected exactly 24 positive relation IDs.')
full_manifest = json.loads((RELEX_RUN / 'manifest.json').read_text())
full_by_pair = {tuple(b['pair_ids']): b for b in full_manifest}
if len(full_manifest) != 36266 or len(full_by_pair) != 36266:
    raise RuntimeError('Unexpected full-validation bag inventory.')
if sum(len(b['sentences']) for b in full_manifest) != 44751:
    raise RuntimeError('Selected sentence inventory changed.')
for bag in full_manifest:
    if not 1 <= len(bag['sentences']) <= 20 or len(bag['entity_positions']) != len(bag['sentences']):
        raise RuntimeError('Invalid selected input inventory.')
    for text, spans in zip(bag['sentences'], bag['entity_positions']):
        for role in ('head', 'tail'):
            start, end = spans[role]
            if (type(start) is not int or type(end) is not int or
                    not 0 <= start < end <= len(text) or text[start:end] != bag[role]):
                raise RuntimeError(f'Invalid supplied {role} span: {bag["pair_ids"]}')
    if set(bag['ds_positive_labels']) - ALLOWED:
        raise RuntimeError('Unknown distant relation label.')

random_inputs = json.loads((RANDOM_100 / 'manifest.json').read_text())
random_settings = json.loads((RANDOM_100 / 'settings.json').read_text())
if (len(random_inputs) != 100 or random_settings['seed'] != 20261006 or
        random_settings['source_manifest_sha256'] != file_digest(RELEX_RUN / 'manifest.json') or
        random_settings['run_fingerprint'] != frozen['fingerprint']):
    raise RuntimeError('Fixed random100 provenance changed.')
expected_random = random.Random(20261006).sample(
    sorted(full_manifest, key=lambda b: tuple(b['pair_ids'])), 100)
if random_inputs != [input_only(b) for b in expected_random]:
    raise RuntimeError('Random100 does not reproduce the prepared sample.')

reviewed_gold, reviewed_id, reference_hashes = {}, {}, {}
for relative in ('support/validation_clean_annotations_2026-09-29.json',
                 'support/validation_expanded_18_annotations_2026-10-01.json'):
    annotation_path = ROOT / relative
    annotation = json.loads(annotation_path.read_text())
    if annotation.get('review_status') != 'human_confirmed' or not annotation.get('annotation_version'):
        raise RuntimeError(f'Annotations are not confirmed/versioned: {relative}')
    sample_path = ROOT / annotation['sample_file']
    if file_digest(sample_path) != annotation['sample_sha256']:
        raise RuntimeError('Reviewed source sample changed.')
    samples = json.loads(sample_path.read_text())['items']
    sample_map = {item['sample_id']: item for item in samples}
    if len(sample_map) != len(samples) or len(annotation['annotations']) != len(samples):
        raise RuntimeError('Incomplete reviewed annotation inventory.')
    checked_ids = set()
    for item in annotation['annotations']:
        source = sample_map[item['sample_id']]
        pair = tuple(source['pair_ids'])
        labels = item.get('relations', item.get('proposed_relations'))
        if (pair in reviewed_gold or item['sample_id'] in checked_ids or
                item['pair_ids'] != source['pair_ids'] or pair not in full_by_pair or
                not isinstance(labels, list) or any(not isinstance(x, str) for x in labels) or
                len(labels) != len(set(labels)) or set(labels) - ALLOWED or item.get('scorable') is False):
            raise RuntimeError('Invalid or duplicate reviewed annotation.')
        bag = full_by_pair[pair]
        if (source['head'] != bag['head'] or source['tail'] != bag['tail'] or
                Counter(source['sentences']) != Counter(bag['sentences'])):
            raise RuntimeError('Reviewed evidence differs from frozen API/encoder input.')
        reviewed_gold[pair] = set(labels)
        reviewed_id[pair] = item['sample_id']
        checked_ids.add(item['sample_id'])
    if checked_ids != set(sample_map):
        raise RuntimeError('Reviewed annotation coverage differs from its sample.')
    reference_hashes[relative] = file_digest(annotation_path)
    reference_hashes[annotation['sample_file']] = file_digest(sample_path)
expected_ids = {f'C{i:02d}' for i in range(1, 13)} | {f'N{i:02d}' for i in range(1, 19)}
if len(reviewed_gold) != 30 or set(reviewed_id.values()) != expected_ids:
    raise RuntimeError('Expected the existing confirmed C01–C12/N01–N18 cohort.')

random_pairs = [tuple(b['pair_ids']) for b in random_inputs]
reviewed_pairs = sorted(reviewed_gold, key=lambda p: reviewed_id[p])
COHORTS = {'reviewed30': reviewed_pairs, 'random100_distant': random_pairs}
selected_pairs = list(dict.fromkeys(reviewed_pairs + random_pairs))
if SCOPE == 'full_validation':
    selected_pairs = [tuple(b['pair_ids']) for b in full_manifest]
    COHORTS['full_validation_distant'] = selected_pairs
SELECTED = [full_by_pair[pair] for pair in selected_pairs]
REFERENCE_METADATA = {'source_hashes': reference_hashes,
                      'labels_sha256': digest({reviewed_id[p]: sorted(v) for p, v in reviewed_gold.items()})}
print(f'Prepared {len(SELECTED):,} distinct bags; reviewed30 and random100 remain separate.')
print('Confirmed positive label instances:', sum(map(len, reviewed_gold.values())))

Prepared 130 distinct bags; reviewed30 and random100 remain separate.
Confirmed positive label instances: 24


## 3. Shared prompt, parsing and positive-label scoring

The request contains head/tail names and each selected sentence with its exact head/tail spans. Distant labels, confirmed labels, split-membership fields and encoder predictions are excluded. The prompt fixes direction, the 24-ID inventory and empty-set abstention. Direct evidence and text-supported linguistic inference are distinguished from unsupported external knowledge.

The positive JSON example specifies syntax only; it contains no example sentence or reference answer. A flat string array is required explicitly, including for positive outputs. It is shared by every mode and does not introduce ICL demonstrations.

The final `content` must contain one JSON object with only `relations`. Duplicate known labels are deduplicated and counted; duplicate JSON keys, unknown IDs, wrong types, empty content, extra keys and non-stop finishes are invalid. No repair prompt or automatic regeneration is used. Invalid responses contribute an empty positive set to end-to-end scoring, with invalidity reported separately. TP/FP/FN and micro-F1 are computed over positive relation sets; valid-empty responses are abstentions, not failures.

In [10]:
SYSTEM_PROMPT = (
    'Extract relations from the supplied news sentences for the ordered HEAD -> TAIL pair. '
    'Entity mentions are supplied; do not detect or substitute other entities. '
    'Character spans are zero-based half-open positions [start,end) in the exact sentence. '
    'Preserve head-to-tail direction even when the tail occurs first in the text. '
    'Use all supplied sentences. Accept directly expressed relations and linguistic or pragmatic '
    'inferences justified by the supplied text. Do not add relations from unsupported outside '
    'knowledge, mere co-occurrence, or an assumed database fact. Multiple relations may be supported. '
    'Return exactly one JSON object with only the key "relations". Its value must be a flat '
    'list of distinct allowed relation-ID strings, never a dictionary, a list of objects or nested lists. '
    'Syntax-only positive example: {"relations": ["/people/person/children"]}. '
    'This example illustrates the output format only; it supplies no evidence about the current pair. '
    'Do not copy the example label unless the supplied sentences support it. '
    'Return {"relations": []} when no allowed positive relation is supported. '
    'Never output NA, an invented label, Markdown, explanations, confidence values or extra keys. '
    'List each accepted relation once. Allowed relations and their head-to-tail meaning:\n'
    + '\n'.join(f'{label}: {LABEL_DESCRIPTIONS[label]}' for label in sorted(ALLOWED))
)

def messages_for(bag):
    evidence = {'head': bag['head'], 'tail': bag['tail'], 'sentences': [
        {'text': text, 'head_span': pos['head'], 'tail_span': pos['tail']}
        for text, pos in zip(bag['sentences'], bag['entity_positions'])]}
    return [{'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': canonical(evidence)}]

def no_duplicate_keys(pairs):
    obj = {}
    for key, value in pairs:
        if key in obj:
            raise ValueError('Repeated JSON object key.')
        obj[key] = value
    return obj

def parse_answer(content, finish_reason):
    if finish_reason != 'stop':
        return {'status': 'invalid', 'relations': [], 'duplicates': 0,
                'error': f'finish_reason={finish_reason}'}
    try:
        if not isinstance(content, str) or not content.strip():
            raise ValueError('Empty or non-string content.')
        obj = json.loads(content, object_pairs_hook=no_duplicate_keys)
        if not isinstance(obj, dict) or set(obj) != {'relations'}:
            raise ValueError('Expected only the relations key.')
        labels = obj['relations']
        if not isinstance(labels, list) or any(not isinstance(x, str) for x in labels):
            raise ValueError('Relations must be a list of strings.')
        if set(labels) - ALLOWED:
            raise ValueError('Unknown or NA relation ID.')
        unique = sorted(set(labels))
        return {'status': 'valid' if unique else 'valid_empty', 'relations': unique,
                'duplicates': len(labels) - len(unique), 'error': None}
    except (ValueError, TypeError) as exc:
        return {'status': 'invalid', 'relations': [], 'duplicates': 0, 'error': str(exc)}

def positive_metrics(gold, predictions, pairs):
    tp = fp = fn = exact = negative_fp_bags = negatives = 0
    for pair in pairs:
        g, p = set(gold[pair]), set(predictions[pair])
        tp += len(g & p); fp += len(p - g); fn += len(g - p)
        exact += g == p
        negatives += not g
        negative_fp_bags += bool(p) and not g
    return {'bags': len(pairs), 'TP': tp, 'FP': fp, 'FN': fn,
            'precision': tp / (tp + fp) if tp + fp else 0.0,
            'recall': tp / (tp + fn) if tp + fn else 0.0,
            'micro_f1': 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0,
            'exact_match_bags': exact, 'reference_empty_bags': negatives,
            'positive_predictions_on_reference_empty_bags': negative_fp_bags}

MODE_SPECS = {}
if RUN_NON_THINKING:
    MODE_SPECS['non_thinking'] = {'thinking': {'type': 'disabled'}, 'reasoning_effort': 'none'}
if RUN_THINKING_LOW:
    MODE_SPECS['thinking_low'] = {'thinking': {'type': 'enabled'}, 'reasoning_effort': 'low'}
if RUN_THINKING_HIGH:
    MODE_SPECS['thinking_high'] = {'thinking': {'type': 'enabled'}, 'reasoning_effort': 'high'}
print('Enabled configurations:', ', '.join(MODE_SPECS))
print('Selected requests:', len(SELECTED) * len(MODE_SPECS), '| total model-usage budget: $', MAX_BUDGET_USD)

def request_for(bag, mode):
    return {'model': MODEL_ID, 'messages': messages_for(bag),
            'response_format': {'type': 'json_object'}, 'stream': False,
            'max_tokens': MAX_COMPLETION_TOKENS, **MODE_SPECS[mode]}

Enabled configurations: non_thinking, thinking_low, thinking_high
Selected requests: 390 | total model-usage budget: $ 2.0


## 4. Off-peak scheduling and usage-based cost estimates

Published peak periods are **01:00–04:00 and 06:00–10:00 UTC, Monday–Friday**. Other periods and UTC weekends are off-peak. Chinese public-holiday exemptions are not modeled, so the guard is conservative. Europe/Rome conversion uses the timezone database and handles daylight-saving changes automatically.

At UTC+2 (including 6 October), weekday off-peak periods in Italy are **00:00–03:00, 06:00–08:00 and 12:00–24:00**. At UTC+1 they become **00:00–02:00, 05:00–07:00 and 11:00–24:00**. Off-peak-only scheduling is disabled by default, so requests can start immediately in either tariff window. If `OFF_PEAK_ONLY=True` and `WAIT_FOR_OFF_PEAK=True`, requests wait in short interruptible intervals during peak periods or within the boundary buffer. This controls submission time; server execution can still cross a boundary.

Costs use provider-reported cache hits/misses and total completion tokens. Reasoning tokens are already included in completion tokens and are not charged twice. A conservative peak rate is assigned to requests crossing a peak boundary. These are reconstructed token charges, not an invoice. A second common uncached/off-peak estimate removes observed cache/time discounts when comparing configurations. Unknown charges retain a conservative reserve.

In [11]:
def is_peak(moment):
    utc = moment.astimezone(timezone.utc)
    return utc.weekday() < 5 and (1 <= utc.hour < 4 or 6 <= utc.hour < 10)

def interval_has_peak(start, end):
    if end < start:
        raise ValueError('Invalid request interval.')
    cursor = start.replace(minute=0, second=0, microsecond=0)
    while cursor <= end:
        if is_peak(cursor):
            return True
        cursor += timedelta(hours=1)
    return is_peak(start) or is_peak(end)

def off_peak_ready(moment):
    return not interval_has_peak(moment, moment + timedelta(seconds=BOUNDARY_BUFFER_SECONDS))

def wait_for_window():
    last_notice = 0.0
    while OFF_PEAK_ONLY and not off_peak_ready(datetime.now(timezone.utc)):
        if not WAIT_FOR_OFF_PEAK:
            raise RunStopped('Outside the buffered off-peak window; resume later.')
        if time.monotonic() - last_notice >= 60:
            print('Waiting for off-peak:', datetime.now(timezone.utc).astimezone(ROME).isoformat(timespec='seconds'))
            last_notice = time.monotonic()
        time.sleep(30)

def normalized_usage(raw):
    if not isinstance(raw, dict):
        raise ValueError('Missing API token usage.')
    def integer(value, name):
        if type(value) is not int or value < 0:
            raise ValueError(f'Invalid usage field: {name}')
        return value
    prompt = integer(raw.get('prompt_tokens'), 'prompt_tokens')
    completion = integer(raw.get('completion_tokens'), 'completion_tokens')
    hit = raw.get('prompt_cache_hit_tokens', (raw.get('prompt_tokens_details') or {}).get('cached_tokens'))
    hit = 0 if hit is None else integer(hit, 'cached_tokens')
    miss = raw.get('prompt_cache_miss_tokens', prompt - hit)
    miss = integer(miss, 'prompt_cache_miss_tokens')
    if hit + miss != prompt:
        raise ValueError('Cache-hit/miss counts do not sum to prompt tokens.')
    details = raw.get('completion_tokens_details') or {}
    reasoning = details.get('reasoning_tokens')
    if reasoning is not None:
        reasoning = integer(reasoning, 'reasoning_tokens')
        if reasoning > completion:
            raise ValueError('Reasoning exceeds total completion tokens.')
    if raw.get('total_tokens', prompt + completion) != prompt + completion:
        raise ValueError('Total usage inventory differs.')
    return {'prompt_tokens': prompt, 'cache_hit_tokens': hit, 'cache_miss_tokens': miss,
            'completion_tokens': completion, 'reasoning_tokens': reasoning,
            'final_tokens': completion - reasoning if reasoning is not None else None}

def usage_cost(usage, tariff, uncached=False):
    rate = RATES[tariff]
    input_cost = (usage['prompt_tokens'] * rate['input_miss'] if uncached else
                  usage['cache_miss_tokens'] * rate['input_miss'] + usage['cache_hit_tokens'] * rate['input_hit'])
    return (input_cost + usage['completion_tokens'] * rate['output']) / 1_000_000

now = datetime.now(timezone.utc)
display(pd.DataFrame([
    {'season': 'UTC+2 / daylight saving', 'weekday_off_peak_Rome': '00–03; 06–08; 12–24'},
    {'season': 'UTC+1 / standard time', 'weekday_off_peak_Rome': '00–02; 05–07; 11–24'},
]))
print('Current Europe/Rome time:', now.astimezone(ROME).isoformat(timespec='seconds'))
print('Current tariff window:', 'peak' if is_peak(now) else 'off_peak')
display(pd.DataFrame([{'tariff': k, **v} for k, v in RATES.items()]))

# Illustrative usage scenarios; no request or tokenizer download is performed.
budget_rows = []
for name, count, input_tokens in [('selected scope', len(SELECTED), len(SELECTED) * 552),
                                  ('full validation', 36266, 20_000_000)]:
    for extra in (0, 1000, 3000):
        usage = {'prompt_tokens': input_tokens, 'cache_miss_tokens': input_tokens,
                 'cache_hit_tokens': 0, 'completion_tokens': count * (80 + extra)}
        budget_rows.append({'scope': name, 'bags': count,
                            'assumed_reasoning_tokens_per_bag': extra,
                            'one_configuration_off_peak_USD': usage_cost(usage, 'off_peak')})
display(pd.DataFrame(budget_rows).round(4))

,season,weekday_off_peak_Rome
0,UTC+2 / daylight saving,00–03; 06–08; 12–24
1,UTC+1 / standard time,00–02; 05–07; 11–24


Current Europe/Rome time: 2026-10-07T10:58:34+02:00
Current tariff window: peak


,tariff,input_miss,input_hit,output
0,off_peak,0.15,0.003,0.6
1,peak,0.30,0.006,1.2


,scope,bags,assumed_reasoning_tokens_per_bag,one_configuration_off_peak_USD
0,selected scope,130,0,0.0170
1,selected scope,130,1000,0.0950
2,selected scope,130,3000,0.2510
3,full validation,36266,0,4.7408
4,full validation,36266,1000,26.5004
5,full validation,36266,3000,70.0196


## 5. Durable API cache and request execution

An inference fingerprint binds the prompt, API settings, frozen full-input manifest and notebook implementation. Scope, display options and annotation labels are excluded, allowing pilot results to be reused in full validation and rescored after confirmed annotation revisions.

Each attempt is journaled before transmission and stores returned content, provider-returned reasoning, usage, timing, model ID and backend fingerprint. Atomic writes and an exclusive local lock control resume. Completed invalid outputs are retained rather than regenerated. Explicit transient HTTP errors have bounded retries; transport timeouts and interruptions stop the run because their charge/completion state is unknown. Pending or uncertain attempts are not automatically resubmitted. The following run-status section lists them for account-side verification before any explicit recovery.

The budget guard uses observed usage charges plus conservative reserves for unknown charges. Before each new request it reserves a byte-based input upper estimate and the configured completion maximum at peak rates. This is a guard on reconstructed model token charges, excluding payment fees/taxes. All attempts, including invalid responses and failed retries, remain in the ledger.

In [12]:
class RunStopped(RuntimeError):
    pass

def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile('w', dir=path.parent, encoding='utf-8', delete=False) as handle:
        temp = Path(handle.name)
        json.dump(value, handle, ensure_ascii=False, sort_keys=True, allow_nan=False)
        handle.write('\n'); handle.flush(); os.fsync(handle.fileno())
    os.replace(temp, path)

def credential():
    key = os.environ.get('DEEPSEEK_API_KEY', '').strip()
    if not key and (ROOT / '.env').is_file():
        for line in (ROOT / '.env').read_text().splitlines():
            name, separator, value = line.strip().removeprefix('export ').partition('=')
            if separator and name.strip() == 'DEEPSEEK_API_KEY':
                values = shlex.split(value, comments=True)
                if len(values) != 1:
                    raise ValueError('DEEPSEEK_API_KEY must be a single .env value.')
                key = values[0].strip()
    if not key or key.lower() in {'...', 'your_api_key', 'your-key-here'}:
        key = getpass('DeepSeek API key (hidden input): ').strip()
    if not key:
        raise RunStopped('No API key supplied.')
    return key

def implementation_hash():
    notebook = json.loads(NOTEBOOK_PATH.read_text())
    ids = {'frozen-inputs', 'prompt-parser', 'tariff-helpers', 'cache-api'}
    sources = {c['id']: ''.join(c['source']) for c in notebook['cells'] if c.get('id') in ids}
    if set(sources) != ids:
        raise RuntimeError('Notebook implementation cells are missing.')
    return digest(sources)

INFERENCE_SETTINGS = {
    'protocol': PROTOCOL_VERSION, 'price_version': PRICE_VERSION,
    'api_url': API_URL, 'requested_model': MODEL_ID, 'advertised_release': ADVERTISED_RELEASE,
    'system_prompt': SYSTEM_PROMPT, 'label_descriptions': LABEL_DESCRIPTIONS,
    'max_tokens': MAX_COMPLETION_TOKENS, 'mode_specs': MODE_SPECS,
    'source_manifest_sha256': file_digest(RELEX_RUN / 'manifest.json'),
    'source_inference_fingerprint': frozen['fingerprint'],
    'notebook_implementation_sha256': implementation_hash(),
    'sampling': 'provider defaults; no temperature/top_p override',
    'duplicate_rule': 'deduplicate known labels and count repeats',
    'invalid_rule': 'empty positive set; no generation retry',
}
INFERENCE_FP = digest(INFERENCE_SETTINGS)
API_RUN = ROOT / 'data/pilot_runs' / RUN_DIRECTORY_NAME / INFERENCE_FP[:16]
ATTEMPTS = API_RUN / 'attempts'
RESULTS = API_RUN / 'bags'

def pair_slug(pair):
    return digest(list(pair))[:24]

def result_path(pair, mode):
    return RESULTS / mode / f'{pair_slug(pair)}.json'

def read_result(bag, mode):
    path = result_path(tuple(bag['pair_ids']), mode)
    if not path.is_file():
        return None
    row = json.loads(path.read_text())
    if (row.get('inference_fingerprint') != INFERENCE_FP or row.get('mode') != mode or
            row.get('pair_ids') != bag['pair_ids'] or
            row.get('request_sha256') != digest(request_for(bag, mode))):
        raise RuntimeError(f'Stale API cache: {path}')
    if row.get('status') not in {'valid', 'valid_empty', 'invalid', 'http_error'}:
        raise RuntimeError(f'Unexpected terminal API status: {path}')
    if row['status'] != 'http_error':
        response = row['response']
        parsed = parse_answer(response['choices'][0]['message'].get('content'),
                              response['choices'][0].get('finish_reason'))
        if any(row[key] != parsed[key] for key in ('status', 'relations', 'duplicates', 'error')):
            raise RuntimeError(f'API parser/cache mismatch: {path}')
    elif row.get('relations') != []:
        raise RuntimeError('An HTTP failure must not invent predictions.')
    attempt_path = ATTEMPTS / f'{row['attempt_id']}.json'
    if not attempt_path.is_file():
        raise RuntimeError('A saved API result has no durable billing journal.')
    attempt = json.loads(attempt_path.read_text())
    if any(attempt.get(k) != row.get(k) for k in ('inference_fingerprint', 'pair_ids', 'mode', 'request_sha256')):
        raise RuntimeError('Result and billing journal identities differ.')
    if row['status'] != 'http_error':
        if attempt.get('state') != 'completed' or attempt.get('response') != row['response']:
            raise RuntimeError('Result differs from its journaled response.')
        normalized_usage(row['response'].get('usage'))
    elif attempt.get('state') != 'http_error':
        raise RuntimeError('HTTP failure differs from its journal state.')
    return row

def read_ledger():
    ledger = {}
    for path in ATTEMPTS.glob('*.json'):
        row = json.loads(path.read_text())
        if row.get('inference_fingerprint') != INFERENCE_FP:
            raise RuntimeError('Attempt ledger fingerprint mismatch.')
        ledger[row['attempt_id']] = row
    return ledger

def ledger_budget(ledger):
    return sum(row.get('cost_usd_estimate', row['reserve_usd']) for row in ledger.values())

def save_attempt(ledger, row):
    atomic_json(ATTEMPTS / f'{row["attempt_id"]}.json', row)
    ledger[row['attempt_id']] = row

@contextlib.contextmanager
def exclusive_run():
    import fcntl
    API_RUN.mkdir(parents=True, exist_ok=True)
    with (API_RUN / 'run.lock').open('a') as handle:
        try:
            fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError:
            raise RunStopped('Another notebook process already owns this API run.')
        try:
            yield
        finally:
            fcntl.flock(handle, fcntl.LOCK_UN)

@contextlib.contextmanager
def awake_context():
    process = None
    if KEEP_AWAKE and sys.platform == 'darwin' and shutil.which('caffeinate'):
        process = subprocess.Popen(['caffeinate', '-i', '-w', str(os.getpid())],
                                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    try:
        yield
    finally:
        if process is not None:
            process.terminate(); process.wait(timeout=10)

def completion_record(bag, mode, response, attempt_id, elapsed):
    choices = response.get('choices')
    if not isinstance(choices, list) or len(choices) != 1 or not isinstance(choices[0].get('message'), dict):
        raise ValueError('Unexpected completion envelope.')
    returned_model = response.get('model')
    if not isinstance(returned_model, str) or returned_model.lower() not in {
            'deepseek-flash', 'deepseek-v4.1-flash'}:
        raise ValueError('The returned model ID does not identify the requested Flash service.')
    usage = normalized_usage(response.get('usage'))
    if MODE_SPECS[mode]['thinking']['type'] == 'disabled' and (
            choices[0]['message'].get('reasoning_content') or (usage['reasoning_tokens'] or 0) > 0):
        raise ValueError('The non-thinking condition returned reasoning; stop and verify API controls.')
    parsed = parse_answer(choices[0]['message'].get('content'), choices[0].get('finish_reason'))
    return {'inference_fingerprint': INFERENCE_FP, 'pair_ids': bag['pair_ids'], 'mode': mode,
            'request_sha256': digest(request_for(bag, mode)), 'attempt_id': attempt_id,
            'wall_s': elapsed, 'response': response, **parsed}

def call_once(bag, mode, key, ledger):
    payload = request_for(bag, mode)
    raw_payload = canonical(payload).encode('utf-8')
    # Byte count deliberately overestimates text tokens; reserve the peak rate.
    input_upper = len(raw_payload) + 512
    reserve = (input_upper * RATES['peak']['input_miss'] +
               MAX_COMPLETION_TOKENS * RATES['peak']['output']) / 1_000_000
    if ledger_budget(ledger) + reserve > MAX_BUDGET_USD:
        raise RunStopped('Model-usage budget reached; increase it explicitly to continue.')
    wait_for_window()
    if shutil.disk_usage(ROOT).free < 256 * 1024**2:
        raise RunStopped('Less than 256 MiB free disk space.')
    started = datetime.now(timezone.utc)
    row = {'attempt_id': uuid.uuid4().hex, 'inference_fingerprint': INFERENCE_FP,
           'pair_ids': bag['pair_ids'], 'mode': mode, 'request_sha256': digest(payload),
           'started_utc': started.isoformat(), 'state': 'inflight', 'reserve_usd': reserve}
    save_attempt(ledger, row)
    request = urllib.request.Request(API_URL, data=raw_payload, method='POST', headers={
        'Authorization': f'Bearer {key}', 'Content-Type': 'application/json'})
    tick = time.monotonic()
    try:
        with urllib.request.urlopen(request, timeout=REQUEST_TIMEOUT_SECONDS) as handle:
            raw_response = handle.read()
        ended = datetime.now(timezone.utc)
        elapsed = time.monotonic() - tick
        response = json.loads(raw_response)
        row['response'] = response
        usage = normalized_usage(response.get('usage'))
        tariff = 'peak' if interval_has_peak(started, ended) else 'off_peak'
        row.update(state='completed', ended_utc=ended.isoformat(), wall_s=elapsed,
                   response=response, usage=usage, tariff_estimate=tariff,
                   cost_usd_estimate=usage_cost(usage, tariff),
                   common_uncached_offpeak_usd=usage_cost(usage, 'off_peak', uncached=True))
        save_attempt(ledger, row)
        result = completion_record(bag, mode, response, row['attempt_id'], elapsed)
        atomic_json(result_path(tuple(bag['pair_ids']), mode), result)
        return result, None
    except urllib.error.HTTPError as exc:
        # Store a sanitized error body, never Authorization headers or the API key.
        body = exc.read(4096).decode('utf-8', errors='replace').replace(key, '[REDACTED]')
        row.update(state='http_error', ended_utc=datetime.now(timezone.utc).isoformat(),
                   wall_s=time.monotonic() - tick, http_status=exc.code, error_body=body)
        save_attempt(ledger, row)  # Unknown billing retains the conservative reserve.
        return None, row
    except BaseException:
        # Unknown completion/charge state must not trigger an automatic duplicate call.
        if row['state'] != 'completed':
            row.update(state='uncertain', ended_utc=datetime.now(timezone.utc).isoformat(),
                       wall_s=time.monotonic() - tick)
            save_attempt(ledger, row)
        raise

def call_with_retries(bag, mode, key, ledger):
    for retry in range(MAX_HTTP_RETRIES + 1):
        result, error = call_once(bag, mode, key, ledger)
        if result is not None:
            return result
        retryable = error['http_status'] in {429, 502, 503, 504}
        if retryable and retry < MAX_HTTP_RETRIES:
            time.sleep(min(2 ** (retry + 1), 30))
            continue
        failure = {'inference_fingerprint': INFERENCE_FP, 'pair_ids': bag['pair_ids'],
                   'mode': mode, 'request_sha256': digest(request_for(bag, mode)),
                   'attempt_id': error['attempt_id'], 'wall_s': error['wall_s'],
                   'status': 'http_error', 'relations': [], 'duplicates': 0,
                   'error': f'HTTP {error["http_status"]}'}
        atomic_json(result_path(tuple(bag['pair_ids']), mode), failure)
        if error['http_status'] in {400, 401, 402, 403, 404, 422}:
            raise RunStopped(f'HTTP {error["http_status"]}; check credentials/balance/API settings before continuing.')
        return failure

def recover_completed_journal(ledger):
    # Restore a result after a successful response was journaled but the result write was interrupted.
    for row in ledger.values():
        if row['state'] != 'completed':
            continue
        bag = full_by_pair[tuple(row['pair_ids'])]
        if result_path(tuple(row['pair_ids']), row['mode']).exists():
            continue
        result = completion_record(bag, row['mode'], row['response'], row['attempt_id'], row['wall_s'])
        atomic_json(result_path(tuple(row['pair_ids']), row['mode']), result)

def mode_order(index):
    # Rotate the first mode so each configuration occupies every request position.
    modes = list(MODE_SPECS)
    shift = index % len(modes)
    return modes[shift:] + modes[:shift]

def run_experiment():
    if not RUN_API:
        print('Paid requests disabled. Set RUN_API=True in section 1 to execute the selected scope.')
        return
    with exclusive_run(), awake_context():
        config_path = API_RUN / 'settings.json'
        if config_path.exists() and json.loads(config_path.read_text()) != INFERENCE_SETTINGS:
            raise RuntimeError('Run settings differ from the existing API cache.')
        atomic_json(config_path, INFERENCE_SETTINGS)
        atomic_json(API_RUN / f'selection_{SCOPE}.json', {
            'scope': SCOPE, 'pair_ids': [b['pair_ids'] for b in SELECTED],
            'input_hashes': {pair_slug(b['pair_ids']): digest(messages_for(b)) for b in SELECTED}})
        ledger = read_ledger()
        recover_completed_journal(ledger)
        # Explicitly authorized resubmission retains the original unknown-charge reserve.
        acknowledged = set(ALLOW_RESUBMIT_UNCERTAIN_ATTEMPT_IDS)
        if acknowledged - set(ledger):
            raise RunStopped('An acknowledged attempt ID does not exist in this run.')
        for attempt_id in acknowledged:
            row = ledger[attempt_id]
            if row['state'] in {'inflight', 'uncertain'}:
                row.update(state='acknowledged_unknown',
                           recovery_note='User explicitly permitted resubmission after account-side review; charge unknown.')
                save_attempt(ledger, row)
        unresolved = [r['attempt_id'] for r in ledger.values() if r['state'] in {'inflight', 'uncertain'}]
        if unresolved:
            raise RunStopped('Unresolved charged/completion states exist; inspect the journal before resubmission: '
                             + ', '.join(unresolved[:5]))
        key = credential()
        failures = 0
        for index, bag in enumerate(SELECTED):
            for mode in mode_order(index):
                if read_result(bag, mode) is not None:
                    continue
                row = call_with_retries(bag, mode, key, ledger)
                failures = failures + 1 if row['status'] in {'invalid', 'http_error'} else 0
                pair = tuple(bag['pair_ids'])
                name = reviewed_id.get(pair, pair_slug(pair)[:8])
                print(f'{index + 1}/{len(SELECTED)} {name} | {mode} | {row["status"]} '
                      f'| ledger usage/reserve ${ledger_budget(ledger):.4f}')
                if failures >= STOP_AFTER_CONSECUTIVE_FAILURES:
                    raise RunStopped('Consecutive failures reached the configured stop limit.')
        print('Selected scope completed. API results:', API_RUN)

## 6. Offline checks of parser, scorer, accounting and tariff boundaries

These constructed cases check duplicate handling, invalid JSON/key/ID behavior, positive-label TP/FP/FN, strict completion status, reasoning-inclusive cost accounting, and UTC/daylight-saving boundaries. They make no API calls and do not inspect test data.

In [13]:
one_label = '/people/person/children'
assert parse_answer('{"relations": []}', 'stop')['status'] == 'valid_empty'
assert parse_answer(canonical({'relations': [one_label, one_label]}), 'stop')['duplicates'] == 1
assert parse_answer('{"relations": [], "relations": []}', 'stop')['status'] == 'invalid'
assert parse_answer('{"relations": ["NA"]}', 'stop')['status'] == 'invalid'
assert parse_answer('{"relations": []}', 'length')['status'] == 'invalid'
toy_gold = {('a', 'b'): {one_label}, ('c', 'd'): set()}
toy_pred = {('a', 'b'): {one_label, '/people/person/religion'}, ('c', 'd'): {one_label}}
toy = positive_metrics(toy_gold, toy_pred, list(toy_gold))
assert (toy['TP'], toy['FP'], toy['FN'], toy['micro_f1']) == (1, 2, 0, 0.5)
usage = normalized_usage({'prompt_tokens': 100, 'prompt_cache_hit_tokens': 20,
                          'prompt_cache_miss_tokens': 80, 'completion_tokens': 110,
                          'completion_tokens_details': {'reasoning_tokens': 100}, 'total_tokens': 210})
assert usage['final_tokens'] == 10
assert math.isclose(usage_cost(usage, 'off_peak'), (80 * .15 + 20 * .003 + 110 * .6) / 1e6)
assert is_peak(datetime(2026, 10, 6, 1, 0, tzinfo=timezone.utc))
assert not is_peak(datetime(2026, 10, 6, 4, 0, tzinfo=timezone.utc))
assert not is_peak(datetime(2026, 10, 10, 2, 0, tzinfo=timezone.utc))
assert datetime(2026, 10, 6, 10, 0, tzinfo=timezone.utc).astimezone(ROME).hour == 12
assert datetime(2026, 10, 26, 10, 0, tzinfo=timezone.utc).astimezone(ROME).hour == 11
# All enabled modes receive identical input and completion limits.
assert set(MODE_SPECS) <= {'non_thinking', 'thinking_low', 'thinking_high'}
requests = [request_for(SELECTED[0], mode) for mode in MODE_SPECS]
assert all(r['messages'] == requests[0]['messages'] for r in requests)
assert all(r['max_tokens'] == MAX_COMPLETION_TOKENS for r in requests)
for mode, spec in MODE_SPECS.items():
    expected = {'non_thinking': ('disabled', 'none'),
                'thinking_low': ('enabled', 'low'), 'thinking_high': ('enabled', 'high')}[mode]
    assert (spec['thinking']['type'], spec['reasoning_effort']) == expected
assert parse_answer('{"relations": [["/people/person/children"]]}', 'stop')['status'] == 'invalid'
assert parse_answer('{"relations": [{"relation": "/people/person/children"}]}', 'stop')['status'] == 'invalid'
assert parse_answer('{"relations": ["/people/person/children"]}', 'stop')['status'] == 'valid'
for position in range(len(MODE_SPECS)):
    assert {mode_order(i)[position] for i in range(len(MODE_SPECS))} == set(MODE_SPECS)
print('Offline constructed checks passed; no API requests were made.')

Offline constructed checks passed; no API requests were made.


## 7. Execute or resume the selected three-mode comparison

Enable `RUN_API` in section 1 before executing. The cell handles budget/scheduling/failure stops and interruptions, then allows the following local status/report cells to inspect retained results. Other unexpected exceptions remain visible. A stopped or incomplete run is never labeled complete.

In [14]:
try:
    run_experiment()
except RunStopped as exc:
    print('RUN STOPPED:', exc)
except KeyboardInterrupt:
    print('Interrupted. Completed responses are retained; inspect unresolved journal states below.')
except (urllib.error.URLError, TimeoutError, socket.timeout) as exc:
    print('Transport stopped; inspect the uncertain journal before resubmission:', type(exc).__name__)

1/130 C01 | non_thinking | valid | ledger usage/reserve $0.0002
1/130 C01 | thinking_low | valid | ledger usage/reserve $0.0032
1/130 C01 | thinking_high | valid | ledger usage/reserve $0.0036
2/130 C02 | thinking_low | valid | ledger usage/reserve $0.0056
2/130 C02 | thinking_high | valid | ledger usage/reserve $0.0073
2/130 C02 | non_thinking | valid | ledger usage/reserve $0.0074
3/130 C03 | thinking_high | valid_empty | ledger usage/reserve $0.0081
3/130 C03 | non_thinking | valid | ledger usage/reserve $0.0082
3/130 C03 | thinking_low | valid_empty | ledger usage/reserve $0.0087
4/130 C04 | non_thinking | valid | ledger usage/reserve $0.0089
4/130 C04 | thinking_low | valid | ledger usage/reserve $0.0105
4/130 C04 | thinking_high | invalid | ledger usage/reserve $0.0156
5/130 C05 | thinking_low | valid | ledger usage/reserve $0.0158
5/130 C05 | thinking_high | valid | ledger usage/reserve $0.0170
5/130 C05 | non_thinking | valid | ledger usage/reserve $0.0171
6/130 C06 | thinking_

## 8. Cached results, completion status and cost inventory

Saved final outputs are revalidated against the current parser and exact request hashes. The attempt journal includes all retries and historical pilot calls in this inference run. Model charges and unknown-charge reserves are reported separately; invoice/payment costs may differ. Reasoning-token counts remain missing when the provider does not return their breakdown.

Unresolved attempts are shown by ID and state. They are not resubmitted automatically because timeout does not prove that the provider performed no billable work. A successful journaled response can be restored locally without another API call.

After account-side verification, an explicit resubmission can be enabled by listing the displayed IDs in `ALLOW_RESUBMIT_UNCERTAIN_ATTEMPT_IDS` and rerunning. Original attempts and their unknown-charge reserves are retained; this does not assert that those calls were free.

In [15]:
API_RESULTS = {mode: {} for mode in MODE_SPECS}
status_rows = []
for mode in MODE_SPECS:
    for bag in SELECTED:
        result = read_result(bag, mode)
        if result is not None:
            API_RESULTS[mode][tuple(bag['pair_ids'])] = result
    counts = Counter(r['status'] for r in API_RESULTS[mode].values())
    status_rows.append({'configuration': mode, 'expected_bags': len(SELECTED),
                        'saved_bags': len(API_RESULTS[mode]), 'missing_bags': len(SELECTED) - len(API_RESULTS[mode]),
                        'valid': counts['valid'], 'valid_empty': counts['valid_empty'],
                        'invalid': counts['invalid'], 'http_error': counts['http_error'],
                        'truncated_completions': sum(
                            r.get('response', {}).get('choices', [{}])[0].get('finish_reason') == 'length'
                            for r in API_RESULTS[mode].values()),
                        'empty_final_answers': sum(
                            not (r['response']['choices'][0]['message'].get('content') or '').strip()
                            for r in API_RESULTS[mode].values() if 'response' in r),
                        'duplicate_labels_removed': sum(r['duplicates'] for r in API_RESULTS[mode].values())})
display(pd.DataFrame(status_rows))
LEDGER = read_ledger()
cost_rows = []
for mode in MODE_SPECS:
    attempts = [r for r in LEDGER.values() if r['mode'] == mode]
    measured = [r for r in attempts if 'cost_usd_estimate' in r]
    unknown = [r for r in attempts if 'cost_usd_estimate' not in r]
    reasoning = [r['usage']['reasoning_tokens'] for r in measured]
    cost_rows.append({'configuration': mode, 'attempts_in_entire_run': len(attempts),
                      'completed_with_usage': len(measured), 'unknown_charge_attempts': len(unknown),
                      'prompt_tokens': sum(r['usage']['prompt_tokens'] for r in measured),
                      'completion_tokens_including_reasoning': sum(r['usage']['completion_tokens'] for r in measured),
                      'reasoning_tokens_if_all_reported': sum(reasoning) if measured and all(x is not None for x in reasoning) else None,
                      'token_charge_USD_estimate': sum(r['cost_usd_estimate'] for r in measured),
                      'common_uncached_offpeak_USD': sum(r['common_uncached_offpeak_usd'] for r in measured),
                      'unknown_charge_reserve_USD': sum(r['reserve_usd'] for r in unknown),
                      'request_wall_s': sum(r.get('wall_s', 0) for r in attempts)})
display(pd.DataFrame(cost_rows).round(5))
unresolved = [{'attempt_id': r['attempt_id'], 'state': r['state'], 'mode': r['mode'],
               'started_utc': r['started_utc'], 'reserve_USD': r['reserve_usd']}
              for r in LEDGER.values() if r['state'] in {'inflight', 'uncertain'}]
if unresolved:
    display(pd.DataFrame(unresolved))
    print('These attempts require account-side verification before any explicit recovery/resubmission.')
print('Result directory:', API_RUN)

,configuration,expected_bags,saved_bags,missing_bags,valid,valid_empty,invalid,http_error,truncated_completions,empty_final_answers,duplicate_labels_removed
0,non_thinking,130,130,0,52,77,1,0,0,0,0
1,thinking_low,130,130,0,33,96,1,0,1,1,0
2,thinking_high,130,130,0,29,97,4,0,4,3,0


,configuration,attempts_in_entire_run,completed_with_usage,unknown_charge_attempts,prompt_tokens,completion_tokens_including_reasoning,reasoning_tokens_if_all_reported,token_charge_USD_estimate,common_uncached_offpeak_USD,unknown_charge_reserve_USD,request_wall_s
0,non_thinking,130,130,0,91342,1110,NaN,0.01417,0.01437,0,116.25227
1,thinking_low,130,130,0,94592,62659,61613.0,0.08889,0.05178,0,396.89342
2,thinking_high,130,130,0,94592,80231,79221.0,0.10998,0.06233,0,475.51623


Result directory: /Users/alessandro/Progetti/Ensembling/data/pilot_runs/deepseek_v41_flash_validation_v2_20261007/19225a2fa7f4645c


## 9. Separate-reference quality and encoder-cache comparison

Each row uses the entire named cohort or remains explicitly incomplete. Missing API or encoder outputs are not silently dropped. Invalid terminal API responses are empty predictions in end-to-end metrics. The reviewed30 reference and distant-label reference are never pooled.

Base/large/GLiDRE comparisons read existing supplied-span score caches only. Encoder labels use strict score > cutoff. Selected base/large intersections/unions and the two existing three-member majority candidates reuse the same cached predictions and include every required member's recorded inference time. Their timing is local CPU inference, whereas API request time includes service/network latency; these are different measurements. Monetary API cost is not compared directly with unmeasured local electricity/hardware cost.

In [16]:
def encoder_cache(run_dir, slug):
    settings_path = run_dir / 'settings.json'
    if not settings_path.is_file():
        return {}, {}
    settings = json.loads(settings_path.read_text())
    if settings['fingerprint'] != hashlib.sha256(json.dumps(settings['settings'], sort_keys=True).encode()).hexdigest():
        raise RuntimeError('Encoder settings fingerprint mismatch.')
    predictions, times = {}, {}
    for bag in SELECTED:
        pair = tuple(bag['pair_ids'])
        # Reproduce the existing encoder cache filename convention.
        cache_id = hashlib.sha256('|'.join(bag['pair_ids']).encode()).hexdigest()[:20]
        path = run_dir / slug / 'bags' / f'{cache_id}.json'
        if not path.is_file():
            continue
        row = json.loads(path.read_text())
        scores = row.get('relation_score_max', {})
        if (row.get('status') != 'valid' or row.get('run_fingerprint') != settings['fingerprint'] or
                row.get('pair_ids') != bag['pair_ids'] or set(scores) != ALLOWED or
                row.get('selected_indices_zero_based') != bag['selected_indices_zero_based'] or
                row.get('sentences_processed') != len(bag['sentences']) or
                any(not isinstance(v, (int, float)) or not math.isfinite(v) or not 0 <= v <= 1 for v in scores.values())):
            raise RuntimeError(f'Invalid or mismatched encoder cache: {path}')
        evidence = row.get('evidence', [])
        if len(evidence) != len(bag['sentences']):
            raise RuntimeError('Incomplete encoder sentence inventory.')
        for index, e in enumerate(evidence):
            if (e['selected_sentence_index'] != index or
                    e['source_record_index'] != bag['selected_indices_zero_based'][index] or
                    e['head_char_span'] != bag['entity_positions'][index]['head'] or
                    e['tail_char_span'] != bag['entity_positions'][index]['tail'] or
                    set(e['relation_scores']) != ALLOWED):
                raise RuntimeError('Encoder supplied-entity evidence changed.')
        if any(not math.isclose(max(e['relation_scores'][label] for e in evidence), scores[label],
                                rel_tol=0, abs_tol=1e-7) for label in ALLOWED):
            raise RuntimeError('Encoder maxima differ from sentence scores.')
        predictions[pair] = scores
        times[pair] = row.get('total_attempt_wall_s', row['wall_s'])
    return predictions, times

SYSTEMS, SYSTEM_TIMES = {}, {}
attempt_times = {mode: defaultdict(float) for mode in MODE_SPECS}
for attempt in LEDGER.values():
    attempt_times[attempt['mode']][tuple(attempt['pair_ids'])] += attempt.get('wall_s', 0)
for mode, rows in API_RESULTS.items():
    SYSTEMS[f'DeepSeek/{mode}'] = {p: set(r['relations']) for p, r in rows.items()}
    SYSTEM_TIMES[f'DeepSeek/{mode}'] = {p: attempt_times[mode][p] for p in rows}

if COMPARE_ENCODER_CACHES:
    encoder_members = {}
    for run, slug, name in ((RELEX_RUN, 'gliner_relex_base', 'relex_base'),
                            (RELEX_RUN, 'gliner_relex_large', 'relex_large'),
                            (GLIDRE_RUN, 'glidre_large', 'glidre_large')):
        score_cache, member_times = encoder_cache(run, slug)
        for cutoff in ENCODER_THRESHOLDS:
            key = f'{name}@{cutoff}'
            SYSTEMS[key] = {p: {label for label, value in scores.items() if value > cutoff}
                            for p, scores in score_cache.items()}
            SYSTEM_TIMES[key] = member_times
            encoder_members[(name, cutoff)] = key
    for base_cutoff, large_cutoff in ((0.5, 0.5), (0.5, 0.7)):
        a = encoder_members[('relex_base', base_cutoff)]
        b = encoder_members[('relex_large', large_cutoff)]
        common = set(SYSTEMS[a]) & set(SYSTEMS[b])
        for fusion in ('intersection', 'union'):
            key = f'{fusion}/base{base_cutoff}/large{large_cutoff}'
            SYSTEMS[key] = {p: SYSTEMS[a][p] & SYSTEMS[b][p] if fusion == 'intersection'
                            else SYSTEMS[a][p] | SYSTEMS[b][p] for p in common}
            SYSTEM_TIMES[key] = {p: SYSTEM_TIMES[a][p] + SYSTEM_TIMES[b][p] for p in common}

# Evaluate the two existing three-member majority candidates on the same API bags.
# Each directed label requires two votes; inference time includes all three members.
    for cutoffs in ((0.5, 0.5, 0.5), (0.5, 0.7, 0.7)):
        members = [encoder_members[(name, cutoff)] for name, cutoff in zip(
            ('relex_base', 'relex_large', 'glidre_large'), cutoffs, strict=True)]
        common = set.intersection(*(set(SYSTEMS[member]) for member in members))
        key = 'majority_2_of_3/base{}/large{}/glidre{}'.format(*cutoffs)
        SYSTEMS[key] = {pair: {label for label in ALLOWED
                               if sum(label in SYSTEMS[member][pair] for member in members) >= 2}
                        for pair in common}
        SYSTEM_TIMES[key] = {pair: sum(SYSTEM_TIMES[member][pair] for member in members)
                             for pair in common}

QUALITY_ROWS = []
for cohort, pairs in COHORTS.items():
    gold = reviewed_gold if cohort == 'reviewed30' else {
        p: set(full_by_pair[p]['ds_positive_labels']) for p in pairs}
    for system, predictions in SYSTEMS.items():
        missing = set(pairs) - set(predictions)
        row = {'reference': cohort, 'system': system, 'expected_bags': len(pairs),
               'missing_bags': len(missing), 'complete': not missing}
        if not missing:
            row.update(positive_metrics(gold, predictions, pairs))
            row['recorded_inference_or_request_wall_s'] = sum(SYSTEM_TIMES[system][p] for p in pairs)
        QUALITY_ROWS.append(row)
quality_table = pd.DataFrame(QUALITY_ROWS)
for cohort in COHORTS:
    print('Reference:', cohort)
    display(quality_table[quality_table['reference'] == cohort].round(4))

Reference: reviewed30


,reference,system,expected_bags,missing_bags,complete,bags,TP,FP,FN,precision,recall,micro_f1,exact_match_bags,reference_empty_bags,positive_predictions_on_reference_empty_bags,recorded_inference_or_request_wall_s
0,reviewed30,DeepSeek/non_thinking,30,0,True,30,19,6,5,0.7600,0.7917,0.7755,20,10,6,30.1027
1,reviewed30,DeepSeek/thinking_low,30,0,True,30,21,2,3,0.9130,0.8750,0.8936,25,10,2,108.2813
2,reviewed30,DeepSeek/thinking_high,30,0,True,30,19,1,5,0.9500,0.7917,0.8636,26,10,1,144.7448
3,reviewed30,relex_base@0.5,30,0,True,30,15,12,9,0.5556,0.6250,0.5882,13,10,3,2.2880
4,reviewed30,relex_base@0.7,30,0,True,30,10,4,14,0.7143,0.4167,0.5263,16,10,0,2.2880
5,reviewed30,relex_base@0.9,30,0,True,30,1,0,23,1.0000,0.0417,0.0800,11,10,0,2.2880
6,reviewed30,relex_large@0.5,30,0,True,30,23,15,1,0.6053,0.9583,0.7419,17,10,3,7.5100
7,reviewed30,relex_large@0.7,30,0,True,30,20,10,4,0.6667,0.8333,0.7407,18,10,2,7.5100
8,reviewed30,relex_large@0.9,30,0,True,30,13,1,11,0.9286,0.5417,0.6842,19,10,0,7.5100
9,reviewed30,glidre_large@0.5,30,0,True,30,13,15,11,0.4643,0.5417,0.5000,16,10,0,4.1707


Reference: random100_distant


,reference,system,expected_bags,missing_bags,complete,bags,TP,FP,FN,precision,recall,micro_f1,exact_match_bags,reference_empty_bags,positive_predictions_on_reference_empty_bags,recorded_inference_or_request_wall_s
18,random100_distant,DeepSeek/non_thinking,100,0,True,100,7,23,2,0.2333,0.7778,0.3590,78,91,20,86.1496
19,random100_distant,DeepSeek/thinking_low,100,0,True,100,7,4,2,0.6364,0.7778,0.7000,94,91,4,288.6122
20,random100_distant,DeepSeek/thinking_high,100,0,True,100,7,3,2,0.7000,0.7778,0.7368,95,91,3,330.7714
21,random100_distant,relex_base@0.5,100,0,True,100,3,19,6,0.1364,0.3333,0.1935,78,91,16,8.4360
22,random100_distant,relex_base@0.7,100,0,True,100,1,6,8,0.1429,0.1111,0.1250,86,91,6,8.4360
23,random100_distant,relex_base@0.9,100,0,True,100,0,0,9,0.0000,0.0000,0.0000,91,91,0,8.4360
24,random100_distant,relex_large@0.5,100,0,True,100,6,20,3,0.2308,0.6667,0.3429,80,91,15,27.3600
25,random100_distant,relex_large@0.7,100,0,True,100,5,14,4,0.2632,0.5556,0.3571,83,91,11,27.3600
26,random100_distant,relex_large@0.9,100,0,True,100,2,3,7,0.4000,0.2222,0.2857,90,91,3,27.3600
27,random100_distant,glidre_large@0.5,100,0,True,100,0,3,9,0.0000,0.0000,0.0000,88,91,3,15.4974


## 10. Paired reviewed errors and measured reasoning overhead

The reviewed error view displays all selected sentences and each available API final answer with confirmed labels, false positives, false negatives and raw final JSON. Provider-returned reasoning is saved locally but is not used as an evaluation label or as proof of the model's actual causal process.

The paired view compares non-thinking/low, non-thinking/high and low/high, counting exact-set matches and errors in each direction. Each pairwise comparison requires both modes on all 30 reviewed bags. Comparisons use this repeatedly inspected development cohort for diagnosis, not a representative estimate or a final-test conclusion. Actual token/latency means from random100 supply a conditional full-validation projection, not a guaranteed invoice.

In [17]:
error_items = []
for pair in reviewed_pairs:
    gold = reviewed_gold[pair]
    available = {mode: rows[pair] for mode, rows in API_RESULTS.items() if pair in rows}
    if not available or not any(set(r['relations']) != gold or r['status'] not in {'valid', 'valid_empty'}
                                for r in available.values()):
        continue
    error_items.append(pair)
for pair in error_items[:MAX_REVIEWED_ERRORS_TO_DISPLAY]:
    bag = full_by_pair[pair]
    print('\n' + '=' * 76)
    print(f'{reviewed_id[pair]}: {bag["head"]} -> {bag["tail"]}')
    print('Confirmed relations:', sorted(reviewed_gold[pair]))
    for i, sentence in enumerate(bag['sentences'], 1):
        print(f'[{i}] {sentence}')
    for mode, rows in API_RESULTS.items():
        row = rows.get(pair)
        if row is None:
            print(mode, '| missing')
            continue
        predicted = set(row['relations'])
        content = row.get('response', {}).get('choices', [{}])[0].get('message', {}).get('content')
        print(mode, '|', row['status'], '| predicted:', sorted(predicted))
        print('FP:', sorted(predicted - reviewed_gold[pair]), '| FN:', sorted(reviewed_gold[pair] - predicted))
        print('Raw final answer:', repr(content))

# Compare every pair of enabled modes; missing cases are never dropped.
transition_rows = []
for mode_a, mode_b in itertools.combinations(MODE_SPECS, 2):
    if any(p not in API_RESULTS[m] for p in reviewed_pairs for m in (mode_a, mode_b)):
        print(f'Paired reviewed30 comparison incomplete: {mode_a} / {mode_b}')
        continue
    transitions = Counter()
    for pair in reviewed_pairs:
        a, b = API_RESULTS[mode_a][pair], API_RESULTS[mode_b][pair]
        a_ok = a['status'] in {'valid', 'valid_empty'} and set(a['relations']) == reviewed_gold[pair]
        b_ok = b['status'] in {'valid', 'valid_empty'} and set(b['relations']) == reviewed_gold[pair]
        outcome = ('both_exact' if a_ok and b_ok else 'b_only_exact' if b_ok
                   else 'a_only_exact' if a_ok else 'neither_exact')
        transitions[outcome] += 1
    for outcome in ('both_exact', 'a_only_exact', 'b_only_exact', 'neither_exact'):
        transition_rows.append({'mode_a': mode_a, 'mode_b': mode_b,
                                'transition': outcome, 'bags': transitions[outcome]})
if transition_rows:
    display(pd.DataFrame(transition_rows))

projection_rows = []
random_pair_set = set(random_pairs)
for mode in MODE_SPECS:
    rows = API_RESULTS[mode]
    if any(p not in rows for p in random_pairs):
        continue
    attempts = [r for r in LEDGER.values() if r['mode'] == mode and tuple(r['pair_ids']) in random_pair_set]
    if any('cost_usd_estimate' not in r for r in attempts):
        print(mode, ': projection withheld because some attempted charges are unknown.')
        continue
    usage_attempts = [r for r in attempts if 'usage' in r]
    projection_rows.append({'configuration': mode, 'random_bags': 100,
        'mean_total_output_tokens_per_bag': sum(r['usage']['completion_tokens'] for r in usage_attempts) / 100,
        'mean_request_wall_s_per_bag': sum(r.get('wall_s', 0) for r in attempts) / 100,
        'projected_36266_bags_common_uncached_offpeak_USD':
            sum(r['common_uncached_offpeak_usd'] for r in usage_attempts) / 100 * 36266,
        'projected_sequential_request_hours': sum(r.get('wall_s', 0) for r in attempts) / 100 * 36266 / 3600})
if projection_rows:
    display(pd.DataFrame(projection_rows).round(4))
else:
    print('A measured full-validation projection requires complete random100 results with usage.')


C01: Papua New Guinea -> Port Moresby
Confirmed relations: ['/location/country/capital', '/location/location/contains']
[1] Port Moresby , the capital of Papua New Guinea , has for decades been portrayed in the Western press as a lawless place overrun by ferocious gangs of '' raskols . ''
non_thinking | valid | predicted: ['/location/country/capital']
FP: [] | FN: ['/location/location/contains']
Raw final answer: '{"relations": ["/location/country/capital"]}'
thinking_low | valid | predicted: ['/location/country/capital']
FP: [] | FN: ['/location/location/contains']
Raw final answer: '{"relations":["/location/country/capital"]}'
thinking_high | valid | predicted: ['/location/country/capital']
FP: [] | FN: ['/location/location/contains']
Raw final answer: '{"relations":["/location/country/capital"]}'

C02: Voltaire -> Paris
Confirmed relations: []
[1] Abdellatif Kechiche 's first film , about a young North African immigrant 's adventures in Paris , was called '' It 's Voltaire 's Fault

,mode_a,mode_b,transition,bags
0,non_thinking,thinking_low,both_exact,20
1,non_thinking,thinking_low,a_only_exact,0
2,non_thinking,thinking_low,b_only_exact,5
3,non_thinking,thinking_low,neither_exact,5
4,non_thinking,thinking_high,both_exact,20
5,non_thinking,thinking_high,a_only_exact,0
6,non_thinking,thinking_high,b_only_exact,5
7,non_thinking,thinking_high,neither_exact,5
8,thinking_low,thinking_high,both_exact,24
9,thinking_low,thinking_high,a_only_exact,1


,configuration,random_bags,mean_total_output_tokens_per_bag,mean_request_wall_s_per_bag,projected_36266_bags_common_uncached_offpeak_USD,projected_sequential_request_hours
0,non_thinking,100,7.52,0.8615,4.0037,8.6786
1,thinking_low,100,443.91,2.8861,13.6354,29.0745
2,thinking_high,100,539.41,3.3077,15.7134,33.3215


## 11. Compact report export

Aggregate quality, completion, usage/cost and projection tables are written to a versioned compact JSON file. It records current reference hashes and whether the selected API scope is complete. Raw responses, returned reasoning, request journals and API keys are not exported to the public results directory. No API calls or encoder inference occur in this section.

In [18]:
if any(API_RESULTS.values()):
    selection_sha = digest([b['pair_ids'] for b in SELECTED])
    report_metadata = {'inference_fingerprint': INFERENCE_FP, 'scope': SCOPE,
                       'protocol_version': PROTOCOL_VERSION, 'mode_specs': MODE_SPECS,
                       'system_prompt_sha256': digest(SYSTEM_PROMPT),
                       'selection_sha256': selection_sha, 'reviewed_reference': REFERENCE_METADATA,
                       'price_version': PRICE_VERSION}
    report_id = digest(report_metadata)[:16]
    report = {'metadata': report_metadata,
              'complete_selected_API_scope': all(len(rows) == len(SELECTED) for rows in API_RESULTS.values()),
              'completion': status_rows, 'cost_entire_inference_run': cost_rows,
              'quality': QUALITY_ROWS, 'random100_based_projections': projection_rows,
              'reviewed_pairwise_transitions': transition_rows,
              'limitations': ['Reviewed30 is a selected repeatedly inspected development sample.',
                              'Distant labels are noisy; references are scored separately.',
                              'API wall time and local CPU inference time are different measurements.',
                              'Token charges are reconstructed estimates, not invoices.',
                              'The API service alias can change; returned model/backend IDs are retained.']}
    public_report = ROOT / 'results' / f'deepseek_flash_{SCOPE}_summary_{report_id}.json'
    atomic_json(public_report, report)
    print('Compact report:', public_report)
else:
    print('No API results exist; no empty performance report was published.')

Compact report: /Users/alessandro/Progetti/Ensembling/results/deepseek_flash_pilot_summary_6b356cadbc56715d.json


## 12. Previous prompt version: historical reference

The original two-mode pilot remains available from its compact report. This table requires identical selected bags and current annotation hashes. It labels prompt versions explicitly: cross-version changes cannot be attributed solely to reasoning effort. No previous result is overwritten, repaired or pooled into the new run.


In [19]:
# Read the original report only; no historical response is regenerated.
if PREVIOUS_PILOT_REPORT.is_file():
    previous = json.loads(PREVIOUS_PILOT_REPORT.read_text())
    metadata = previous['metadata']
    same_inputs = metadata['selection_sha256'] == digest([b['pair_ids'] for b in SELECTED])
    same_reference = metadata['reviewed_reference'] == REFERENCE_METADATA
    if same_inputs and same_reference:
        history_rows = []
        for version, rows in (('v1 original prompt', previous['quality']),
                              ('v2 explicit JSON format', QUALITY_ROWS)):
            for row in rows:
                if row['system'].startswith('DeepSeek/'):
                    history_rows.append({'prompt_version': version, **row})
        display(pd.DataFrame(history_rows).round(4))
        print('Only within-v2 mode comparisons hold the prompt constant.')
    else:
        print('Historical comparison withheld: selected bags or current annotations differ.')
else:
    print('The original compact pilot report is not present in this checkout.')


,prompt_version,FN,FP,TP,bags,complete,exact_match_bags,expected_bags,micro_f1,missing_bags,positive_predictions_on_reference_empty_bags,precision,recall,recorded_inference_or_request_wall_s,reference,reference_empty_bags,system
0,v1 original prompt,13,2,11,30,True,18,30,0.5946,0,2,0.8462,0.4583,27.0015,reviewed30,10,DeepSeek/non_thinking
1,v1 original prompt,4,2,20,30,True,25,30,0.8696,0,2,0.9091,0.8333,131.2572,reviewed30,10,DeepSeek/thinking_high
2,v1 original prompt,8,1,1,100,True,91,100,0.1818,0,1,0.5000,0.1111,85.8185,random100_distant,91,DeepSeek/non_thinking
3,v1 original prompt,3,4,6,100,True,93,100,0.6316,0,4,0.6000,0.6667,345.9589,random100_distant,91,DeepSeek/thinking_high
4,v2 explicit JSON format,5,6,19,30,True,20,30,0.7755,0,6,0.7600,0.7917,30.1027,reviewed30,10,DeepSeek/non_thinking
5,v2 explicit JSON format,3,2,21,30,True,25,30,0.8936,0,2,0.9130,0.8750,108.2813,reviewed30,10,DeepSeek/thinking_low
6,v2 explicit JSON format,5,1,19,30,True,26,30,0.8636,0,1,0.9500,0.7917,144.7448,reviewed30,10,DeepSeek/thinking_high
7,v2 explicit JSON format,2,23,7,100,True,78,100,0.3590,0,20,0.2333,0.7778,86.1496,random100_distant,91,DeepSeek/non_thinking
8,v2 explicit JSON format,2,4,7,100,True,94,100,0.7000,0,4,0.6364,0.7778,288.6122,random100_distant,91,DeepSeek/thinking_low
9,v2 explicit JSON format,2,3,7,100,True,95,100,0.7368,0,3,0.7000,0.7778,330.7714,random100_distant,91,DeepSeek/thinking_high


Only within-v2 mode comparisons hold the prompt constant.
